# General

For more informations, si the documentation *Documentary Strategy*.

# Import & Configs

In [14]:
import json
import pandas as pd

from sentence_transformers import SentenceTransformer
import chromadb

# Upload Data

In [3]:
def get_chunks_df():
    chunks = []
    
    with open("../data/chunks/chunks.jsonl", "r") as f:
    
        for line in f:
    
            chunks.append(json.loads(line))

    return pd.DataFrame(chunks)

In [4]:
chunk_df =  get_chunks_df()

In [5]:
chunk_df.head()

,chunk_id,pmid,chunk_index,title,year,text
0,42200192_0,42200192,0,Prediction of treatment failure in patients wi...,2026,Prediction of treatment failure in patients wi...
1,42200192_1,42200192,1,Prediction of treatment failure in patients wi...,2026,This study aimed to identify predictors of ear...
2,42200192_2,42200192,2,Prediction of treatment failure in patients wi...,2026,Treatment failure was defined as tumor progres...
3,42200192_3,42200192,3,Prediction of treatment failure in patients wi...,2026,RESULTS: Among 62 patients diagnosed between 0...
4,42200192_4,42200192,4,Prediction of treatment failure in patients wi...,2026,"A model, including age, perfusion metrics, and..."


# Embedding 
## Model

In [8]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

/home/jeremy/Documents/dev/LLM_RAG/Medical_assistant/.ma_env/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19639.65it/s]


## Test

In [9]:
sample_embedding = embedding_model.encode(
    chunk_df.iloc[0]["text"]
)

In [10]:
sample_embedding.shape

(384,)

## Chunk Transformation

In [11]:
texts = chunk_df["text"].tolist()

In [12]:
embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

Batches: 100%|██████████| 102/102 [04:32<00:00,  2.67s/it]


In [13]:
embeddings.shape

(3260, 384)

# VectorDB
## Client

In [15]:
client = chromadb.PersistentClient(
    path="../vectorstore/chroma_db"
)

## Collection

In [32]:
#client.delete_collection("medical_rag")

In [16]:
collection = client.get_or_create_collection(
    name="medical_rag"
)

In [17]:
collection.add(

    ids=chunk_df["chunk_id"].tolist(),

    documents=chunk_df["text"].tolist(),

    embeddings=embeddings.tolist(),

    metadatas=[

        {
            "pmid": row["pmid"],
            "title": row["title"],
            "year": int(row["year"])
        }

        for _, row in chunk_df.iterrows()
    ]
)

## Retrieval Test

In [22]:
def test_query(query):
    query_embedding = embedding_model.encode(
        query
    )
    
    results = collection.query(
    
        query_embeddings=[query_embedding.tolist()],
    
        n_results=3
    )

    for i, doc in enumerate(results["documents"][0]):
    
        print(f"\nRESULT {i+1}")
        print(doc[:500])

In [29]:
queries = [
    "MRI diagnosis of glioblastoma",
    
    "brain tumor MRI",

    "glioblastoma prognosis",

    "tumor progression",

    "brain edema",

    "radiotherapy treatment"
]

In [30]:
for query in queries:
    print(f'{query:-^50}')
    test_query(query)
    print("\n\n\n")

----------MRI diagnosis of glioblastoma-----------

RESULT 1
Impact on survival of glioblastoma patient's in relation to the imaging of the peri-surgical area: a multi-parametric diffusion MRI, perfusion MRI and [11C]MET PET study. OBJECTIVES: Glioblastoma (GBM) is an aggressive brain tumour with poor prognosis; recurrence near the surgical cavity is common despite multimodal-treatment. Conventional MRI lacks accuracy for early detection of recurrence and cannot reliably differentiate progression from pseudoprogression.

RESULT 2
Gliomas are the most common primary malignant brain tumors and are characterized by heterogeneous growth and complex biology, which complicate accurate diagnosis and management. While magnetic resonance imaging (MRI) remains the clinical standard, its limitations in delineating tumor margins and distinguishing recurrence from treatment-induced changes highlight the need for complementary molecular imaging.

RESULT 3
METHODS: Clinical 3-Tesla brain MRI images f

# Dowload Data

*Chroma* already saves all the data in *../vectorstore/chroma_db*. If needed, use the code below to save the data.

In [31]:
#chunk_df.to_parquet(
#    "../data/processed/chunk_metadata.parquet"
#)